## Random Hill Climbing for the Knapsack problem

### imports and global variables

In [ ]:
import random, os

FILES_TO_TEST = [
    {"name": "../a1/knapsack-20.txt", "size": 20},
    {"name": "../a1/knapsack-200.txt", "size": 200}
]

EVALUATIONS = 1000

### helpers

In [2]:
def load_data(file_name: str) -> tuple[list[tuple[int, int]], int]:
    """
    Loads weights, values, and max capacity from the file.
    Returns a tuple: (list_of_items, max_capacity)
    """
    items = []
    capacity = 0

    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return [], 0

    with open(file_name) as f:
        lines = f.readlines()

        for line in lines[:-1]:
            parts = line.strip().split()
            if len(parts) >= 3:
                weight = int(parts[1])
                value = int(parts[2])
                items.append((weight, value))

        try:
            capacity = int(lines[-1].strip())
        except ValueError:
            print("Error parsing capacity from last line.")

    return items, capacity


def evaluate_solution(
    solution: list[int], items: list[tuple[int, int]], max_weight: int
) -> tuple[int, int]:
    """
    Calculates total weight and value of a solution.
    Returns (weight, value).
    If weight > max_weight, value is set to 0 (penalty for infeasibility).
    """
    total_weight = 0
    total_value = 0

    for i, is_included in enumerate(solution):
        if is_included == 1:
            total_weight += items[i][0]
            total_value += items[i][1]

    # Check feasibility
    if total_weight > max_weight:
        return total_weight, 0  # Infeasible solution

    return total_weight, total_value


def get_random_solution(n: int) -> list[int]:
    """
    Generates a single random binary list of length n.
    This replaces the inefficient get_lists recursive function.
    """
    return [random.randint(0, 1) for _ in range(n)]

def N(l: list[int]) -> list[list[int]]:
    ''' 
    returns all neighbours by flipping every possible bit and returns a list of lists
    '''
    ans = []
    for i in range(len(l)):
        ans.append(l[:i] + [1-l[i]] + l[i+1:])
    return ans

def get_neighbour(l: list[int], i: int) -> list[int]:
    '''
    flips bit at locus i on a list and returns the new list
    ''' 
    return l[:i] + [1-l[i]] + l[i+1:]

### Core Algorithm

In [33]:
def rhc(items: list[tuple[int,int]], max_weight: int, k: int) -> tuple[list[int],int, int]: 
    ''' 
    performs random hill-climbing while evaluating k points
    returns the best string found
    '''
    
    n = len(items)

    
    current = get_random_solution(n)
    current_weight, current_value = evaluate_solution(current, items, max_weight)    
    for _ in range(k-1):
        i = random.randint(0,n-1)
    
        candidate = get_neighbour(current, i)
        
        w, v = evaluate_solution(candidate, items, max_weight)
        
        if v >= current_value:
            current = candidate 
            current_value = v 
            current_weight = w
    
    return current, current_value, current_weight

### Running the experiment

In [53]:
def run():
    for file_info in FILES_TO_TEST:
        filename = file_info['name']
        items, capacity = load_data(filename)
        
        if not items:
            continue
        
        s, v, w = rhc(items, capacity, EVALUATIONS)
        print(f"solution for file {filename}: {s}\n with value {v} \n and weight {w}")

if __name__ == '__main__':
    run()

solution for file ../a1/knapsack-20.txt: [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1]
 with value 579 
 and weight 521
solution for file ../a1/knapsack-200.txt: [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0]
 with value 96948 
 and weight 112648


# A2 - Next Ascent Hill Climbing

### Imports and Global variables

In [4]:
import random, os, time

FILES_TO_TEST = [
    {"name": "../a1/knapsack-20.txt", "size": 20},
    {"name": "../a1/knapsack-200.txt", "size": 200}
]

EVALUATIONS = [10, 50, 100, 500, 1000]

NUM_RUNS = 25

OUTPUT_FILE = "output.md"


### Helpers

In [1]:

def load_data(file_name: str) -> tuple[list[tuple[int, int]], int]:
    """
    Loads weights, values, and max capacity from the file.
    Returns a tuple: (list_of_items, max_capacity)
    """
    items = []
    capacity = 0

    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return [], 0

    with open(file_name) as f:
        lines = f.readlines()

        for line in lines[:-1]:
            parts = line.strip().split()
            if len(parts) >= 3:
                weight = int(parts[1])
                value = int(parts[2])
                items.append((weight, value))

        try:
            capacity = int(lines[-1].strip())
        except ValueError:
            print("Error parsing capacity from last line.")

    return items, capacity


def evaluate_solution(
    solution: list[int], items: list[tuple[int, int]], max_weight: int
) -> tuple[int, int]:
    """
    Calculates total weight and value of a solution.
    Returns (weight, value).
    If weight > max_weight, value is set to 0 (penalty for infeasibility).
    """
    total_weight = 0
    total_value = 0

    for i, is_included in enumerate(solution):
        if is_included == 1:
            total_weight += items[i][0]
            total_value += items[i][1]

    # Check feasibility
    if total_weight > max_weight:
        return total_weight, 0  # Infeasible solution

    return total_weight, total_value


def get_random_solution(n: int) -> list[int]:
    """
    Generates a single random binary list of length n.
    This replaces the inefficient get_lists recursive function.
    """
    return [random.randint(0, 1) for _ in range(n)]

def get_neighbour(l: list[int], i: int) -> list[int]:
    '''
    flips bit at locus i on a list and returns the new list
    ''' 
    return l[:i] + [1-l[i]] + l[i+1:]
    
    

### Core Algorithm

In [2]:
def nahc(items: list[tuple[int, int]], max_weight: int, max_evals: int) -> dict: 
    '''
    Next Ascent Hill-Climbing.
    1. Random string (current-hilltop).
    2. Iterate bits. If fitness increases, accept and continue scanning from next bit.
    3. If no increase found in whole pass, restart with random string.
    '''
    n = len(items)
    
    best_overall_val = -1
    best_overall_sol = []
    best_overall_w = 0
    
    evals_used = 0
    
    while evals_used < max_evals:
        # Step 1: Choose a string at random (Restart)
        current_sol = get_random_solution(n)
        current_w, current_v = evaluate_solution(current_sol, items, max_weight)
        evals_used += 1
        
        # Update global best check
        if current_v > best_overall_val:
            best_overall_val = current_v
            best_overall_sol = current_sol
            best_overall_w = current_w
        
        # Step 2: Systematic scan
        # Start scanning from bit 0 (or wherever logic dictates, usually 0 on restart)
        i = 0 
        
        while i < n and evals_used < max_evals:
            # Flip bit i
            neighbor = get_neighbour(current_sol, i)
            nw, nv = evaluate_solution(neighbor, items, max_weight)
            evals_used += 1
            
            # Update global best check
            if nv > best_overall_val:
                best_overall_val = nv
                best_overall_sol = neighbor
                best_overall_w = nw

            # If fitness increase, keep new string
            if nv > current_v:
                current_sol = neighbor
                current_v = nv
                current_w = nw
                # "Continue mutating the new string starting immediately after the bit position"
                # We don't break; we just continue the loop with i+1
            
            i += 1
            
        # Step 3: If no increases found (loop finished without improvement), 
        # the outer while loop handles the restart.
            
    return {"value": best_overall_val, "weight": best_overall_w, "solution": best_overall_sol}

### Running Experiments

In [ ]:
def run_experiments(num_runs=30):
    """
    Runs the NAHC algorithm multiple times for each parameter setting,
    aggregates results, and writes them to a markdown table file.
    """    
    # Open file and write the header
    with open(OUTPUT_FILE, "w") as f:
        f.write(f"| {'Instance':<20} | {'Runs':<6} | {'K (Evaluations)':<15} | {'Best Value (Max)':<18} | {'Avg Best Value':<15} | {'Avg Time (s)':<12} |\n")
        f.write("|---|---|---|---|---|---|\n")

    for file_info in FILES_TO_TEST:
        filename = file_info["name"]
        items, capacity = load_data(filename)

        if not items:
            with open(OUTPUT_FILE, "a") as f:
                f.write(f"| {filename} | {'N/A':<6} | {'N/A':<15} | {'Error: File not found':<18} | {'N/A':<15} | {'N/A':<12} |\n")
            continue

        # Iterate through each evaluation limit (K)
        for k in EVALUATIONS:
            best_overall_value = -1
            sum_best_values = 0
            total_time = 0

            # Perform multiple independent runs for the same setting
            for run in range(num_runs):
                start_time = time.time()
                result = nahc(items, capacity, k)
                elapsed = time.time() - start_time
                
                total_time += elapsed
                sum_best_values += result['value']
                
                if result['value'] > best_overall_value:
                    best_overall_value = result['value']

            # Calculate averages across the runs
            avg_value = sum_best_values / num_runs
            avg_time = total_time / num_runs

            # Save results of the experiments to the output file
            with open(OUTPUT_FILE, "a") as f:
                f.write(
                    f"| {filename:<20} | {num_runs:<6} | {k:<15} | {best_overall_value:<18} | {avg_value:<15.2f} | {avg_time:<12.4f} |\n"
                )

if __name__ == '__main__':
    run_experiments(NUM_RUNS)